# Notebook 3: Interventions and Qualitative Analysis

Colab-ready workflow for running held-out residual-patch interventions on candidate circuits and interpreting the results.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/jacobposchl/model-interpretability.git'
REPO_DIR = Path('/content/model_interpretability' if IN_COLAB else Path.cwd()).resolve()

if IN_COLAB:
    if REPO_DIR.exists():
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '-q'], check=True)

    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/flow_circuits')
    DATA_ROOT  = Path('/content/drive/MyDrive/ctls/data')
else:
    if not (REPO_DIR / 'flow_circuits').exists():
        raise RuntimeError('Run this notebook from the repository root or in Google Colab.')
    os.chdir(REPO_DIR)
    DRIVE_ROOT = REPO_DIR / 'artifacts' / 'local_notebook_runtime'
    DATA_ROOT  = DRIVE_ROOT / 'data'

BACKBONE_ROOT   = DRIVE_ROOT / 'backbones'
EXPERIMENT_ROOT = DRIVE_ROOT / 'experiments'
NOTEBOOK_ROOT   = DRIVE_ROOT / 'notebook_runs'
for path in (DRIVE_ROOT, DATA_ROOT, BACKBONE_ROOT, EXPERIMENT_ROOT, NOTEBOOK_ROOT):
    path.mkdir(parents=True, exist_ok=True)

CONFIG_ROOT = REPO_DIR / 'configs' / 'flow'
print(f"Repo      : {REPO_DIR}")
print(f"Drive root: {DRIVE_ROOT}")
print(f"Data      : {DATA_ROOT}")

In [ ]:
from flow_circuits.data import build_cifar10_splits
from flow_circuits.interventions import run_circuit_interventions
from flow_circuits.training import collect_model_outputs, load_components_from_checkpoint

## Step 1 ? Configure Your Comparison Run

Edit the cell below. This notebook never retrains the flow model.

---

**`TRAINING_MODE`** ? controls how much discovery/intervention work runs:

| Mode | Workload | Use for |
|------|----------|---------|
| `'smoke'` | very small subsets | quick pipeline check |
| `'debug'` | medium subsets | faster comparison |
| `'full'` | config defaults | full intervention comparison |

---

**`CONFIG_NAME`** ? should point to an aligned experiment that already produced both `phase_b.pt` and `phase_c.pt`.

---

**Path overrides** ? set these only if your checkpoints or comparison artifacts live outside the canonical experiment directory.


In [ ]:
# ?? Comparison mode ???????????????????????????????????????????????????????????
#   'smoke' = tiny discovery/intervention subsets (fast pipeline check)
#   'debug' = medium subsets (faster comparison)
#   'full'  = full discovery/intervention settings from the config
TRAINING_MODE = 'full'  # <- change me

# ?? Model configuration ???????????????????????????????????????????????????????
#   Use an aligned config that already has both phase_b.pt and phase_c.pt
CONFIG_NAME = 'resnet18_aligned'  # <- change me

# ?? Optional path overrides ???????????????????????????????????????????????????
CHECKPOINT_PATH = None  # directory containing phase_b.pt and phase_c.pt
CIRCUITS_PATH   = None  # directory for phase_b/phase_c circuit + intervention artifacts

# ?? Output directory ??????????????????????????????????????????????????????????
OUTPUT_DIR = str(NOTEBOOK_ROOT / 'nb03')


In [ ]:
import copy
import json
import queue
import shlex
import subprocess
import sys
import threading
import time
from pathlib import Path

import matplotlib.pyplot as plt
import yaml

_MODE_SETTINGS = {
    'smoke': {
        'label': 'Smoke comparison - tiny discovery/intervention subsets',
        'disc_images': 256,
        'disc_bootstrap': 4,
        'interv_images': 128,
    },
    'debug': {
        'label': 'Debug comparison - medium discovery/intervention subsets',
        'disc_images': 2000,
        'disc_bootstrap': 10,
        'interv_images': 1000,
    },
    'full': {
        'label': 'Full comparison - config discovery/intervention settings',
        'disc_images': None,
        'disc_bootstrap': None,
        'interv_images': None,
    },
}
_ms = _MODE_SETTINGS[TRAINING_MODE]
print(f"Comparison mode : {TRAINING_MODE}")
print(f"  {_ms['label']}")

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLI_MODULES = {
    'flow-discover': 'flow_circuits.cli.discover',
    'flow-intervene': 'flow_circuits.cli.intervene',
}

def _format_seconds(seconds: float) -> str:
    seconds = int(max(0, round(seconds)))
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours}h {minutes}m {secs}s"
    if minutes:
        return f"{minutes}m {secs}s"
    return f"{secs}s"


def _timestamp() -> str:
    return time.strftime('%H:%M:%S')


def _log_step(step_label: str, message: str) -> None:
    print(f"[{_timestamp()}] {step_label}: {message}", flush=True)


def _display_limit(value: int | None) -> str:
    return 'all available' if value is None else f'{value:,}'


def run_flow_cli(command: str, *args: str, step_label: str, heartbeat_seconds: int = 30) -> None:
    def _stream(cmd: list[str]) -> None:
        import os

        env = os.environ.copy()
        env['PYTHONUNBUFFERED'] = '1'
        display_cmd = shlex.join(cmd)
        _log_step(step_label, 'starting command')
        _log_step(step_label, display_cmd)
        start_time = time.time()
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            cwd=REPO_DIR,
            env=env,
            bufsize=1,
        )
        output_queue: queue.Queue[str | None] = queue.Queue()

        def _pump_stdout() -> None:
            assert process.stdout is not None
            for line in process.stdout:
                output_queue.put(line)
            output_queue.put(None)

        threading.Thread(target=_pump_stdout, daemon=True).start()
        stream_closed = False
        next_heartbeat = start_time + heartbeat_seconds
        while not stream_closed or process.poll() is None:
            try:
                line = output_queue.get(timeout=1.0)
            except queue.Empty:
                now = time.time()
                if now >= next_heartbeat:
                    _log_step(step_label, f"still running... elapsed {_format_seconds(now - start_time)}")
                    next_heartbeat = now + heartbeat_seconds
                continue
            if line is None:
                stream_closed = True
                continue
            print(line, end='', flush=True)
            next_heartbeat = time.time() + heartbeat_seconds

        returncode = process.wait()
        elapsed = time.time() - start_time
        if returncode != 0:
            _log_step(step_label, f"failed after {_format_seconds(elapsed)}")
            raise subprocess.CalledProcessError(returncode, cmd)
        _log_step(step_label, f"completed in {_format_seconds(elapsed)}")

    try:
        _stream([command, *args])
    except FileNotFoundError:
        _log_step(step_label, f"`{command}` not found on PATH; retrying via `python -m`")
        _stream([sys.executable, '-m', CLI_MODULES[command], *args])

with open(str(CONFIG_ROOT / f'{CONFIG_NAME}.yaml'), 'r', encoding='utf-8') as handle:
    config = yaml.safe_load(handle)

if config['experiment'].get('mode') != 'aligned':
    raise ValueError('Notebook 3 compares phase_b.pt and phase_c.pt, so use an aligned config.')

config['data']['data_dir'] = str(DATA_ROOT)
config['data']['download'] = False
config['logging']['checkpoint_dir'] = str(EXPERIMENT_ROOT / config['experiment']['name'])

def _normalize_dir(value: str | None, expected_suffix: str) -> Path | None:
    if not value:
        return None
    path = Path(value)
    return path.parent if path.suffix == expected_suffix else path

checkpoint_root = _normalize_dir(CHECKPOINT_PATH, '.pt') or Path(config['logging']['checkpoint_dir'])
artifact_root = _normalize_dir(CIRCUITS_PATH, '.json') or checkpoint_root
artifact_root.mkdir(parents=True, exist_ok=True)

effective_config = copy.deepcopy(config)
if TRAINING_MODE != 'full':
    effective_config['discovery']['max_images'] = _ms['disc_images']
    effective_config['discovery']['bootstrap_iterations'] = _ms['disc_bootstrap']
    effective_config['interventions']['max_images'] = _ms['interv_images']

EFFECTIVE_CONFIG = OUTPUT_DIR / f'{TRAINING_MODE}_config.yaml'
EFFECTIVE_CONFIG.write_text(yaml.safe_dump(effective_config, sort_keys=False), encoding='utf-8')

MODEL_ORDER = [('phase_b', 'Phase B'), ('phase_c', 'Phase C')]
MODEL_SPECS = {
    tag: {
        'tag': tag,
        'label': label,
        'checkpoint': checkpoint_root / f'{tag}.pt',
        'circuits': artifact_root / f'candidate_circuits_{tag}.json',
        'summary_json': artifact_root / f'intervention_summary_{tag}.json',
        'summary_csv': artifact_root / f'intervention_summary_{tag}.csv',
    }
    for tag, label in MODEL_ORDER
}

disc_cfg = effective_config['discovery']
interv_cfg = effective_config['interventions']
print()
print(f"Config         : {EFFECTIVE_CONFIG}")
print('Discovery work :')
print(f"  max_images          : {_display_limit(disc_cfg.get('max_images'))}")
print(f"  seeds               : {disc_cfg.get('seeds') or [disc_cfg.get('seed', 0)]}")
print(f"  cluster bootstraps  : {disc_cfg.get('bootstrap_iterations', 20)}")
print(f"  stability bootstrap : {disc_cfg.get('stability_bootstrap_iterations', 500)}")
print('Interventions  :')
print(f"  max_images          : {_display_limit(interv_cfg.get('max_images'))}")
print(f"  alpha               : {interv_cfg.get('alpha', 0.05)}")
for tag, label in MODEL_ORDER:
    spec = MODEL_SPECS[tag]
    print(f"{label} checkpoint : {spec['checkpoint']}")
    print(f"{label} circuits    : {spec['circuits']}")
    print(f"{label} summary     : {spec['summary_json']}")


## Step 2 ? Run Side-By-Side Discovery and Interventions

This notebook never retrains the flow model.

For each checkpoint:
1. verify the checkpoint exists
2. run discovery if its circuit artifact is missing
3. run interventions if its summary is missing
4. compare the saved results side by side


In [ ]:
missing_checkpoints = [
    f"{spec['label']} checkpoint -> {spec['checkpoint']}"
    for spec in MODEL_SPECS.values()
    if not spec['checkpoint'].exists()
]
if missing_checkpoints:
    raise FileNotFoundError(
        'Notebook 3 requires pre-trained phase_b.pt and phase_c.pt checkpoints.\\n'
        + '\\n'.join(missing_checkpoints)
    )

circuits_by_phase = {}
summaries_by_phase = {}
for phase_index, (tag, label) in enumerate(MODEL_ORDER, start=1):
    spec = MODEL_SPECS[tag]
    print()
    print('=' * 72)
    print(f"{label} ({phase_index}/{len(MODEL_ORDER)})")
    print('=' * 72)
    print(f"Checkpoint : {spec['checkpoint']}")
    print(f"Circuits   : {spec['circuits']}")
    print(f"Summary    : {spec['summary_json']}")

    if not spec['circuits'].exists():
        print('Status     : discovery artifact missing; running discovery now.')
        run_flow_cli(
            'flow-discover',
            '--checkpoint',
            str(spec['checkpoint']),
            '--output',
            str(spec['circuits']),
            step_label=f'{label} discovery',
        )
    else:
        print('Status     : discovery artifact already exists; loading saved results.')

    _log_step(f'{label} discovery', 'loading circuit artifact from disk')
    circuits_by_phase[tag] = json.loads(spec['circuits'].read_text(encoding='utf-8'))
    circuit_artifact = circuits_by_phase[tag]
    _log_step(
        f'{label} discovery',
        f"loaded {len(circuit_artifact['circuits'])} candidate circuit(s) from {circuit_artifact['metadata']['n_images']} discovery images",
    )

    if not spec['summary_json'].exists():
        print('Status     : intervention summary missing; running interventions now.')
        run_flow_cli(
            'flow-intervene',
            '--checkpoint',
            str(spec['checkpoint']),
            '--circuits',
            str(spec['circuits']),
            '--output',
            str(spec['summary_json']),
            step_label=f'{label} interventions',
        )
    else:
        print('Status     : intervention summary already exists; loading saved results.')

    _log_step(f'{label} interventions', 'loading intervention summary from disk')
    summaries_by_phase[tag] = json.loads(spec['summary_json'].read_text(encoding='utf-8'))
    results = summaries_by_phase[tag]
    n_validated = sum(1 for row in results if row.get('validated'))
    _log_step(
        f'{label} interventions',
        f'loaded {len(results)} intervention row(s); {n_validated} validated circuit(s)',
    )

print()
for tag, label in MODEL_ORDER:
    results = summaries_by_phase[tag]
    n_validated = sum(1 for row in results if row.get('validated'))
    print(f"{label}: {len(results)} circuit(s) tested, {n_validated} validated")


## Step 3 ? Compare Intervention Results

The cells below compare `phase_b.pt` and `phase_c.pt` on the same discovery and intervention workflow.


In [ ]:
import numpy as np

print(f"{'Model':<8} {'Tested':>7} {'Validated':>10} {'Member drop':>12} {'Control drop':>13} {'Gap':>9}")
print(f"{'-'*8} {'-'*7} {'-'*10} {'-'*12} {'-'*13} {'-'*9}")
for tag, label in MODEL_ORDER:
    results = summaries_by_phase[tag]
    member = [row['mean_member_delta_margin'] for row in results]
    control = [row['mean_nonmember_delta_margin'] for row in results]
    gap = [m - c for m, c in zip(member, control)]
    n_validated = sum(1 for row in results if row.get('validated'))
    print(
        f"{label:<8} {len(results):>7} {n_validated:>10} "
        f"{(np.mean(member) if member else float('nan')):>12.4f} "
        f"{(np.mean(control) if control else float('nan')):>13.4f} "
        f"{(np.mean(gap) if gap else float('nan')):>9.4f}"
    )


In [ ]:
import numpy as np

fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(7 * len(MODEL_ORDER), 4), squeeze=False)
for axis, (tag, label) in zip(axes[0], MODEL_ORDER):
    results = summaries_by_phase[tag]
    if not results:
        axis.text(0.5, 0.5, f'No intervention rows for {label}', ha='center', va='center')
        axis.axis('off')
        continue
    labels = [f"C{row['circuit_id']}" for row in results]
    member = [row['mean_member_delta_margin'] for row in results]
    nonmember = [row['mean_nonmember_delta_margin'] for row in results]
    random_node = [row['mean_random_node_delta_margin'] for row in results]
    random_cell = [row['mean_random_cell_delta_margin'] for row in results]
    x = np.arange(len(labels))
    axis.plot(x, member, marker='o', label='members')
    axis.plot(x, nonmember, marker='o', label='matched non-members')
    axis.plot(x, random_node, marker='o', label='random-node control')
    axis.plot(x, random_cell, marker='o', label='random-cell control')
    axis.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    axis.set_xticks(x)
    axis.set_xticklabels(labels, rotation=45, ha='right')
    axis.set_ylabel('Mean confidence drop (margin delta)')
    axis.set_title(f'{label}: intervention effect by circuit')
    axis.legend()
plt.tight_layout()


In [ ]:
import numpy as np

max_plots = max((min(4, len(circuits_by_phase[tag]['circuits'])) for tag, _ in MODEL_ORDER), default=0)
if max_plots == 0:
    print('No candidate circuits available in either artifact.')
else:
    grid_size = next(iter(circuits_by_phase.values()))['metadata']['grid_size']
    fig, axes = plt.subplots(len(MODEL_ORDER), max_plots, figsize=(4 * max_plots, 3.5 * len(MODEL_ORDER)), squeeze=False)
    for row_idx, (tag, label) in enumerate(MODEL_ORDER):
        circuits = circuits_by_phase[tag]['circuits'][:max_plots]
        for col_idx in range(max_plots):
            axis = axes[row_idx][col_idx]
            if col_idx >= len(circuits):
                axis.axis('off')
                continue
            circuit = circuits[col_idx]
            heatmap = np.zeros((grid_size, grid_size), dtype=float)
            for _, cell_idx in circuit['active_nodes']:
                row = cell_idx // grid_size
                col = cell_idx % grid_size
                heatmap[row, col] += 1.0
            image = axis.imshow(heatmap, cmap='viridis')
            axis.set_title(f"{label} ? C{circuit['id']} footprint")
            plt.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    plt.tight_layout()


In [ ]:
for tag, label in MODEL_ORDER:
    results = summaries_by_phase[tag]
    validated = [row for row in results if row.get('validated')]
    print(label)
    if validated:
        print(f"  {len(validated)} validated circuit(s):")
        for row in validated:
            print(
                f"    C{row['circuit_id']}: members={row['n_members']}, "
                f"member_drop={row['mean_member_delta_margin']:+.4f}, "
                f"nonmember_drop={row['mean_nonmember_delta_margin']:+.4f}"
            )
    else:
        print('  No circuits passed the current intervention validation rule.')
    print()
